# Initialization Strategy Benchmark

Quantitative comparison of three initialization strategies for `optimize_circular_layout_with_enclosed_nodes`:

| Strategy | Description |
|----------|-------------|
| **random** | Uniform random positions, max-leaf radius for non-leaves (current default) |
| **treemap** | Squarified treemap: nested rectangles → circumradii (top-down) |
| **greedy_bu** | Greedy bottom-up packing: leaves placed first, sets wrap children |

**Problems tested:**
- *Multiples diagrams* — sets are "multiples of k" for k ∈ {2,3,4,5,7}; elements are integers 2..N.  
  Structure: one set-to-set nesting (×4 ⊂ ×2) and many partial overlaps (e.g. multiples of 6 are in both ×2 and ×3).
- *DAG multiples* — includes ×6 (child of both ×2 and ×3), creating a true DAG.
- *Balanced trees* — pure nesting, no partial overlaps; treemap is specifically designed for this case.

**Metrics:**
- Pre-GD: fraction of containment edges already satisfied, initial per-term losses.
- Post-GD (2000 iters): final per-term losses, fraction of containment edges satisfied.
- Convergence curves for the three most representative problems.

In [ ]:
import time
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd

from vizopt.base import ObjectiveTerm, OptimizationProblemTemplate, OptimConfig
from vizopt.templates.nested_circles import (
    treemap_node_positions,
    greedy_bottomup_node_positions,
    _compute_collision_pairs,
    _term_total_size,
    _term_collision,
    _term_non_inclusion,
)

## Problem builders

In [ ]:
def make_multiples_graph(ks, N, leaf_size=1.0):
    """
    Inclusion DiGraph for the Euler diagram of 'multiples of k' sets.

    Set nodes: '×k' for k in ks (non-leaf, will get optimizable radii).
    Leaf nodes: integers in 2..N that are multiples of at least one k.

    Edges use the Hasse diagram convention (transitive edges excluded):
    - Set-to-set: (×k, ×m) if k|m and no intermediate ×l with k|l, l|m exists.
    - Set-to-leaf: (×k, i) if k|i and no ×m with k|m also divides i (innermost set only).
    """
    ks = sorted(set(ks))
    G = nx.DiGraph()
    set_names = {k: f"×{k}" for k in ks}  # ×k
    for name in set_names.values():
        G.add_node(name)

    # Set-to-set: Hasse diagram of divisibility
    for k in ks:
        for m in ks:
            if k < m and m % k == 0:
                has_intermediate = any(
                    mid != k and mid != m and mid % k == 0 and m % mid == 0
                    for mid in ks
                )
                if not has_intermediate:
                    G.add_edge(set_names[k], set_names[m])

    # Set-to-leaf: innermost set for each integer
    for i in range(2, N + 1):
        divs = [k for k in ks if i % k == 0]
        if not divs:
            continue
        innermost = [
            k for k in divs
            if not any(m != k and m % k == 0 and i % m == 0 for m in divs)
        ]
        G.add_node(i, size=float(leaf_size))
        for k in innermost:
            G.add_edge(set_names[k], i)

    return G


def make_balanced_tree(branching, depth, leaf_size=1.0):
    """
    Balanced tree DiGraph with parent→child edges.
    Leaf nodes (deepest level) carry a 'size' attribute.
    depth=1: root + branching leaves.
    depth=2: root + branching internal nodes + branching² leaves.
    """
    G = nx.DiGraph()
    node_id = 0
    G.add_node(0)

    def _add_children(parent, remaining):
        nonlocal node_id
        if remaining == 0:
            return
        for _ in range(branching):
            node_id += 1
            child = node_id
            if remaining == 1:
                G.add_node(child, size=float(leaf_size))
            else:
                G.add_node(child)
            G.add_edge(parent, child)
            _add_children(child, remaining - 1)

    _add_children(0, depth)
    return G

## Benchmark utilities

In [ ]:
def describe_graph(G):
    """Summary statistics for an inclusion graph."""
    n_leaves = sum(1 for n in G.nodes if G.out_degree(n) == 0)
    n_sets   = sum(1 for n in G.nodes if G.out_degree(n) > 0)
    leaf_set = {n for n in G.nodes if G.out_degree(n) == 0}
    set_nodes = [n for n in G.nodes if G.out_degree(n) > 0]
    memberships = {
        n: {d for d in nx.descendants(G, n) if d in leaf_set}
        for n in set_nodes
    }
    n_nested = n_disjoint = n_partial = 0
    for i, a in enumerate(set_nodes):
        for b in set_nodes[:i]:
            la, lb = memberships[a], memberships[b]
            if la <= lb or lb <= la:
                n_nested += 1
            elif la.isdisjoint(lb):
                n_disjoint += 1
            else:
                n_partial += 1
    return {
        "n_leaves":  n_leaves,
        "n_sets":    n_sets,
        "n_edges":   G.number_of_edges(),
        "n_nested":  n_nested,
        "n_partial": n_partial,
        "n_disjoint":n_disjoint,
    }


def random_init(inclusion_graph, canvas_size=1.0, seed=42):
    """Uniform random positions; radius scaled by sqrt(n_leaf_descendants) to avoid gradient instability.

    Using a fixed max-leaf-radius for all non-leaf nodes (like the public API does) causes
    NaN for tree-structured problems: the root must enclose many leaves but starts too small,
    producing enormous non-inclusion gradients that destabilise Adam.
    """
    rng = np.random.default_rng(seed)
    pos = {
        node: (float(rng.uniform(0, canvas_size)), float(rng.uniform(0, canvas_size)))
        for node in inclusion_graph.nodes
    }
    leaf_set = {n for n in inclusion_graph.nodes if inclusion_graph.out_degree(n) == 0}
    max_leaf_r = max(
        (inclusion_graph.nodes[n].get("size", 1.0) for n in leaf_set), default=1.0
    )
    var_radii = {}
    for n in inclusion_graph.nodes:
        if inclusion_graph.out_degree(n) > 0:
            n_desc = max(1, sum(1 for d in nx.descendants(inclusion_graph, n) if d in leaf_set))
            var_radii[n] = float(np.sqrt(n_desc) * max_leaf_r * 1.5)
    return pos, var_radii


def compute_containment_fraction(pos, all_radii, graph):
    """Fraction of inclusion edges where child circle is inside parent circle."""
    slacks = []
    for parent, child in graph.edges():
        px, py = pos[parent]
        cx, cy = pos[child]
        dist = np.sqrt((px - cx)**2 + (py - cy)**2)
        slacks.append(all_radii[parent] - dist - all_radii[child])
    if not slacks:
        return 1.0, 0.0
    slacks = np.array(slacks)
    return float(np.mean(slacks >= -1e-9)), float(slacks.mean())

In [ ]:
def benchmark_problem(inclusion_graph, n_iters=2000, lr=0.005, seed=42, track_every=20):
    """
    Benchmark all three init strategies on the given inclusion graph.
    Returns dict keyed by strategy name, each containing pre/post-GD metrics and history.
    """
    leaf_nodes     = [n for n in inclusion_graph.nodes if inclusion_graph.out_degree(n) == 0]
    non_leaf_nodes = [n for n in inclusion_graph.nodes if inclusion_graph.out_degree(n) > 0]
    all_node_names = leaf_nodes + non_leaf_nodes

    fixed_radii  = np.array([inclusion_graph.nodes[n].get("size", 1.0) for n in leaf_nodes], dtype=np.float32)
    total_scale  = float(fixed_radii.sum()) if len(fixed_radii) else 10.0
    fallback_r   = float(fixed_radii.max()) if len(fixed_radii) else 1.0

    node_to_id       = {name: i for i, name in enumerate(all_node_names)}
    inc_edges        = [(node_to_id[u], node_to_id[v]) for u, v in inclusion_graph.edges()]
    inc_edge_indices = (
        np.array(inc_edges, dtype=np.int32) if inc_edges
        else np.zeros((0, 2), dtype=np.int32)
    )
    collision_pairs = _compute_collision_pairs(all_node_names, inclusion_graph)

    input_params = {
        "fixed_node_radii":      fixed_radii,
        "collision_pairs":       collision_pairs,
        "inclusion_edge_indices": inc_edge_indices,
    }
    terms = [
        ObjectiveTerm("total_size",    _term_total_size,    2.0),
        ObjectiveTerm("collision",     _term_collision,     1.0),
        ObjectiveTerm("non_inclusion", _term_non_inclusion, 1.0),
    ]

    strategies = {
        "random":    lambda: random_init(inclusion_graph, canvas_size=total_scale, seed=seed),
        "treemap":   lambda: treemap_node_positions(inclusion_graph, canvas_size=total_scale),
        "greedy_bu": lambda: greedy_bottomup_node_positions(inclusion_graph, canvas_size=total_scale),
    }

    results = {}
    for strat_name, init_fn in strategies.items():
        pos_init, var_radii_init = init_fn()

        init_xys = np.stack(
            [pos_init.get(n, (total_scale / 2, total_scale / 2)) for n in all_node_names],
            dtype=np.float32,
        )
        init_vr = np.array(
            [var_radii_init.get(n, fallback_r) for n in non_leaf_nodes], dtype=np.float32
        )
        init_ov = {"node_xys": init_xys, "variable_node_radii": init_vr}

        # Pre-GD containment
        all_radii_init = {
            **{leaf_nodes[i]: float(fixed_radii[i]) for i in range(len(leaf_nodes))},
            **{non_leaf_nodes[j]: float(init_vr[j]) for j in range(len(non_leaf_nodes))},
        }
        cont_frac, mean_slack = compute_containment_fraction(pos_init, all_radii_init, inclusion_graph)

        # Pre-GD per-term losses
        init_ts  = float(_term_total_size(init_ov, input_params))
        init_col = float(_term_collision(init_ov, input_params))
        init_ni  = float(_term_non_inclusion(init_ov, input_params))

        # GD run
        _xys_cap = init_xys.copy()
        _vr_cap  = init_vr.copy()

        def _initialize(_, _s, xys=_xys_cap, vr=_vr_cap):
            return {"node_xys": xys.copy(), "variable_node_radii": vr.copy()}

        problem = OptimizationProblemTemplate(terms=terms, initialize=_initialize).instantiate(input_params)
        optim_vars, history = problem.optimize(
            OptimConfig(n_iters=n_iters, learning_rate=lr),
            callback=lambda *_: None
        )

        # Post-GD containment
        pos_final = {
            n: tuple(float(c) for c in xy)
            for n, xy in zip(all_node_names, optim_vars["node_xys"])
        }
        final_vr_dict = {n: float(r) for n, r in zip(non_leaf_nodes, optim_vars["variable_node_radii"])}
        all_radii_final = {
            **{leaf_nodes[i]: float(fixed_radii[i]) for i in range(len(leaf_nodes))},
            **final_vr_dict,
        }
        final_cont_frac, _ = compute_containment_fraction(pos_final, all_radii_final, inclusion_graph)

        results[strat_name] = {
            "init_containment_%": round(cont_frac * 100, 1),
            "init_mean_slack":    round(mean_slack, 3),
            "init_total_size":    round(init_ts,  3),
            "init_collision":     round(init_col, 4),
            "init_non_inclusion": round(init_ni,  4),
            "final_containment_%": round(final_cont_frac * 100, 1),
            "final_total_size":   round(history[-1]["total_size"],    4),
            "final_collision":    round(history[-1]["collision"],     5),
            "final_non_inclusion":round(history[-1]["non_inclusion"], 5),
            "final_total_loss":   round(history[-1]["total"],        4),
            "history":            history,
        }

    return results

## Define problems and inspect their structure

In [ ]:
problems = {
    # Multiples of {2,3,4,5,7} — as requested; one set-to-set nesting (×4 ⊂ ×2)
    "mult [2,3,4,5,7] N=20":   make_multiples_graph([2,3,4,5,7], 20),
    "mult [2,3,4,5,7] N=40":   make_multiples_graph([2,3,4,5,7], 40),
    "mult [2,3,4,5,7] N=70":   make_multiples_graph([2,3,4,5,7], 70),
    # All-prime divisors — no set-to-set nesting, only partial overlaps
    "mult [2,3,5,7] N=30":     make_multiples_graph([2,3,5,7], 30),
    # DAG: ×6 is child of both ×2 and ×3
    "mult [2,3,4,5,6,7] N=40": make_multiples_graph([2,3,4,5,6,7], 40),
    # Pure tree — no partial overlaps; treemap designed for this
    "tree 3-ary depth=2":      make_balanced_tree(3, 2),   # 3 sets + 9 leaves
    "tree 3-ary depth=3":      make_balanced_tree(3, 3),   # 13 nodes + 27 leaves
    "tree 4-ary depth=2":      make_balanced_tree(4, 2),   # 5 sets + 16 leaves
}

print(f"{'Problem':30s}  {'leaves':>6}  {'sets':>4}  {'edges':>5}  {'nested':>6}  {'partial':>7}  {'disjoint':>8}")
print("-" * 80)
for name, G in problems.items():
    d = describe_graph(G)
    print(
        f"{name:30s}  {d['n_leaves']:6d}  {d['n_sets']:4d}  {d['n_edges']:5d}"
        f"  {d['n_nested']:6d}  {d['n_partial']:7d}  {d['n_disjoint']:8d}"
    )

## Run benchmarks

Each problem × 3 strategies = gradient descent run (2000 iters, lr=0.005).  
First JAX call per problem shape triggers tracing (~5–20 s); subsequent calls are fast.

In [ ]:
N_ITERS = 2000
LR = 0.005

all_results = {}
for prob_name, G in problems.items():
    print(f"  {prob_name} ...", end=" ", flush=True)
    t0 = time.perf_counter()
    all_results[prob_name] = benchmark_problem(G, n_iters=N_ITERS, lr=LR)
    print(f"{time.perf_counter() - t0:.1f}s")
print("Done.")

## Results: pre-GD initialization quality

In [ ]:
INIT_COLS = ["init_containment_%", "init_mean_slack", "init_total_size", "init_collision", "init_non_inclusion"]

rows = []
for prob_name, strats in all_results.items():
    d = describe_graph(problems[prob_name])
    for strat, metrics in strats.items():
        rows.append({
            "problem":  prob_name,
            "strategy": strat,
            "n_leaves": d["n_leaves"],
            "n_partial": d["n_partial"],
            **{k: metrics[k] for k in INIT_COLS},
        })

df_init = (
    pd.DataFrame(rows)
    .set_index(["problem", "strategy"])
    .sort_index()
)
display(df_init)

## Results: post-GD quality (2000 iterations)

In [ ]:
FINAL_COLS = ["final_containment_%", "final_total_size", "final_collision", "final_non_inclusion", "final_total_loss"]

rows_f = []
for prob_name, strats in all_results.items():
    for strat, metrics in strats.items():
        rows_f.append({
            "problem":  prob_name,
            "strategy": strat,
            **{k: metrics[k] for k in FINAL_COLS},
        })

df_final = (
    pd.DataFrame(rows_f)
    .set_index(["problem", "strategy"])
    .sort_index()
)
display(df_final)

## Compact pivot: final_total_loss by problem × strategy

In [ ]:
df_pivot_loss = (
    pd.DataFrame(rows_f)[["problem", "strategy", "final_total_loss"]]
    .pivot(index="problem", columns="strategy", values="final_total_loss")
    [["random", "treemap", "greedy_bu"]]
)
# Best strategy per problem
df_pivot_loss["best"] = df_pivot_loss.idxmin(axis=1)
print("Final total loss (lower is better):")
display(df_pivot_loss)

df_pivot_ni = (
    pd.DataFrame(rows_f)[["problem", "strategy", "final_non_inclusion"]]
    .pivot(index="problem", columns="strategy", values="final_non_inclusion")
    [["random", "treemap", "greedy_bu"]]
)
df_pivot_ni["best"] = df_pivot_ni.idxmin(axis=1)
print("\nFinal non-inclusion loss (lower = fewer containment violations):")
display(df_pivot_ni)

df_pivot_cont = (
    pd.DataFrame(rows)[["problem", "strategy", "init_containment_%"]]
    .pivot(index="problem", columns="strategy", values="init_containment_%")
    [["random", "treemap", "greedy_bu"]]
)
df_pivot_cont["best"] = df_pivot_cont.idxmax(axis=1)
print("\nInit containment % (higher = more edges already satisfied at t=0):")
display(df_pivot_cont)

## Convergence curves

Three representative problems: multiples N=40 (with set nesting), DAG multiples, pure tree depth=3.

In [ ]:
PLOT_PROBLEMS = [
    "mult [2,3,4,5,7] N=40",
    "mult [2,3,4,5,6,7] N=40",
    "tree 3-ary depth=3",
]
STRATEGIES = ["random", "treemap", "greedy_bu"]
COLORS  = {"random": "#9E9E9E", "treemap": "#2196F3", "greedy_bu": "#FF5722"}
METRICS = ["total", "total_size", "collision", "non_inclusion"]
YLABELS = ["total loss", "total size (×2)", "collision", "non-inclusion"]

fig, axes = plt.subplots(
    len(PLOT_PROBLEMS), len(METRICS),
    figsize=(17, 3.8 * len(PLOT_PROBLEMS)),
)

for row_i, prob_name in enumerate(PLOT_PROBLEMS):
    strats = all_results[prob_name]
    for col_j, (metric, ylabel) in enumerate(zip(METRICS, YLABELS)):
        ax = axes[row_i, col_j]
        for strat in STRATEGIES:
            h = strats[strat]["history"]
            iters = [e["iteration"] for e in h]
            vals  = [e[metric]     for e in h]
            ax.plot(iters, vals, label=strat, color=COLORS[strat], linewidth=1.8)
        if row_i == 0 and col_j == 0:
            ax.legend(fontsize=9)
        ax.set_title(f"{prob_name}\n{ylabel}", fontsize=9)
        ax.set_xlabel("iteration", fontsize=8)
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

plt.suptitle("Convergence: random vs treemap vs greedy_bu initialization", fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

## Non-inclusion convergence — all problems

In [ ]:
prob_names = list(problems.keys())
ncols = 3
nrows = (len(prob_names) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(13, 3.5 * nrows))
axes_flat = axes.ravel()

for ax_i, prob_name in enumerate(prob_names):
    ax = axes_flat[ax_i]
    strats = all_results[prob_name]
    for strat in STRATEGIES:
        h = strats[strat]["history"]
        iters = [e["iteration"] for e in h]
        vals  = [e["non_inclusion"] for e in h]
        ax.plot(iters, vals, label=strat, color=COLORS[strat], linewidth=1.5)
    ax.set_title(prob_name, fontsize=8)
    ax.set_yscale("log")
    ax.grid(True, alpha=0.3)
    if ax_i == 0:
        ax.legend(fontsize=8)
    if ax_i % ncols == 0:
        ax.set_ylabel("non-inclusion loss")

for ax in axes_flat[len(prob_names):]:
    ax.set_visible(False)

plt.suptitle("Non-inclusion loss over iterations (lower = containment better satisfied)", fontsize=10, y=1.01)
plt.tight_layout()
plt.show()

## "Iterations to threshold" — how fast does containment improve?

For each problem and strategy, find the first iteration where `non_inclusion < threshold`.

In [ ]:
THRESHOLD = 0.05  # adjust if needed based on the loss scale of your problems

rows_t = []
for prob_name, strats in all_results.items():
    for strat, metrics in strats.items():
        h = metrics["history"]
        crossed = next(
            (e["iteration"] for e in h if e["non_inclusion"] < THRESHOLD),
            None,
        )
        rows_t.append({
            "problem":  prob_name,
            "strategy": strat,
            f"iters_to_ni<{THRESHOLD}": crossed if crossed is not None else f">{N_ITERS}",
        })

df_thresh = (
    pd.DataFrame(rows_t)
    .pivot(index="problem", columns="strategy", values=f"iters_to_ni<{THRESHOLD}")
    [["random", "treemap", "greedy_bu"]]
)
print(f"First iteration where non_inclusion < {THRESHOLD} (None = never reached):")
display(df_thresh)